#  CIFAR-10 Image Classification Assignment
## ANN vs CNN — Architecture Comparison & Training Strategy Analysis

**Objective:** Build, train, and compare ANN and CNN models on CIFAR-10. Complete all beginner tasks including advanced training strategies.

---
**CIFAR-10 Classes:** Airplane · Automobile · Bird · Cat · Deer · Dog · Frog · Horse · Ship · Truck

## Step 1 — Imports & Setup

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)

## Step 2 — Load CIFAR-10 Dataset

- **50,000** training images · **10,000** test images
- Each image: **32×32 pixels × 3 color channels (RGB)**

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print("Train shape:", x_train.shape, "| Labels:", y_train.shape)
print("Test  shape:", x_test.shape,  "| Labels:", y_test.shape)
print("Pixel value range:", x_train.min(), "–", x_train.max())

## Step 3 — Visualize Sample Images

In [ ]:
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]], fontsize=9)
    plt.axis('off')
plt.suptitle('CIFAR-10 Sample Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 4 — Preprocessing

Normalize pixel values **0–255 → 0.0–1.0** to:
- Stabilize gradient updates during backpropagation
- Prevent vanishing/exploding gradients
- Speed up convergence

For ANN, images are also **flattened** from (32, 32, 3) → (3072,) since Dense layers expect 1-D input.

In [ ]:
# Normalize
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm  = x_test.astype('float32')  / 255.0

# Flatten for ANN (32*32*3 = 3072)
x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat  = x_test_norm.reshape(len(x_test_norm), -1)

print("Normalized range: [{:.1f}, {:.1f}]".format(x_train_norm.min(), x_train_norm.max()))
print("ANN input shape (flat):", x_train_flat.shape)
print("CNN input shape (spatial):", x_train_norm.shape)

---
#  Part 1: Baseline ANN Model

ANN treats images as **flat vectors** — it loses all spatial information.

### Task 1 Implemented: Increased Dense Layers
Original: 512 → 256 → 10  
Upgraded: **512 → 256 → 128 → 64 → 10** (deeper layout for better feature learning)

In [ ]:
# ── Baseline ANN (original architecture) ──────────────────────────────────
ann_model = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
], name='ANN_Baseline')

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()

In [ ]:
ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64,
    verbose=1
)

In [ ]:
ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test, verbose=0)
print(f"ANN Test Accuracy : {ann_test_acc:.4f}")
print(f"ANN Test Loss     : {ann_test_loss:.4f}")

---
#  Part 2: Baseline CNN Model

CNN uses **Conv2D** to extract local spatial features (edges, textures, shapes).

### Task 2 Implemented: Filter Scaling 32 → 64 → 128
- Conv Block 1: **32 filters** → detects low-level features (edges)
- Conv Block 2: **64 filters** → detects mid-level features (textures)
- Conv Block 3: **128 filters** → detects high-level features (object parts)

In [ ]:
cnn_model = models.Sequential([
    # Block 1 — 32 filters
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 2 — 64 filters
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 3 — 128 filters
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='CNN_Baseline')

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
cnn_history = cnn_model.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64,
    verbose=1
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print(f"CNN Test Accuracy : {cnn_test_acc:.4f}")
print(f"CNN Test Loss     : {cnn_test_loss:.4f}")

---
##  Validation Accuracy Comparison — ANN vs CNN (10 Epochs)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Validation Accuracy
axes[0].plot(ann_history.history['val_accuracy'], marker='o', label='ANN Val Acc', color='#e74c3c', linewidth=2)
axes[0].plot(cnn_history.history['val_accuracy'], marker='s', label='CNN Val Acc', color='#2980b9', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('ANN vs CNN — Validation Accuracy', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(range(10))
axes[0].set_xticklabels(range(1, 11))

# Right: Validation Loss
axes[1].plot(ann_history.history['val_loss'], marker='o', label='ANN Val Loss', color='#e74c3c', linewidth=2)
axes[1].plot(cnn_history.history['val_loss'], marker='s', label='CNN Val Loss', color='#2980b9', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('ANN vs CNN — Validation Loss', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(range(10))
axes[1].set_xticklabels(range(1, 11))

plt.tight_layout()
plt.show()

---
#  Part 3: Data Augmentation CNN

**Why augmentation?** It prevents overfitting by synthetically expanding training data via:
- **RandomFlip** — horizontal mirroring (a cat facing left = cat facing right)
- **RandomRotation** — slight tilts up to ±36°
- **RandomZoom** — zoom in/out by up to 10%

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
], name='DataAugmentation')

aug_cnn_model = models.Sequential([
    data_augmentation,
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='CNN_Augmented')

aug_cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

aug_cnn_model.summary()

In [ ]:
# Task 5: Train the augmented network (executed)
aug_history = aug_cnn_model.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64,
    verbose=1
)

aug_test_loss, aug_test_acc = aug_cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print(f"Augmented CNN Test Accuracy : {aug_test_acc:.4f}")
print(f"Augmented CNN Test Loss     : {aug_test_loss:.4f}")

---
# Student Tasks — All 5 Implemented

## Task 3: Train ANN for 20 Epochs (with Task 1: Increased Dense Layers)

In [ ]:
# Task 1 + Task 3: Deeper ANN layout, trained for 20 epochs
ann_deep = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='ANN_Deep_20epochs')

ann_deep.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Training deeper ANN for 20 epochs...")
ann_deep_history = ann_deep.fit(
    x_train_flat, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    verbose=1
)

ann_deep_loss, ann_deep_acc = ann_deep.evaluate(x_test_flat, y_test, verbose=0)
print(f"Deep ANN (20 epochs) Test Accuracy: {ann_deep_acc:.4f}")

## Task 3: Train CNN for 20 Epochs (with Task 2: 32→64→128 Filters)

In [ ]:
# Task 2 + Task 3: CNN with filter scaling, trained for 20 epochs
cnn_20ep = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='CNN_FilterScaled_20epochs')

cnn_20ep.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Training CNN (32→64→128 filters) for 20 epochs...")
cnn_20ep_history = cnn_20ep.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    verbose=1
)

cnn_20ep_loss, cnn_20ep_acc = cnn_20ep.evaluate(x_test_norm, y_test, verbose=0)
print(f"CNN 20-epoch Test Accuracy: {cnn_20ep_acc:.4f}")

## Task 4: EarlyStopping — CNN with Automatic Stop

**EarlyStopping** monitors `val_loss` and halts training when it stops improving:
- `patience=5`: Wait 5 epochs with no improvement before stopping
- `restore_best_weights=True`: Roll back to the epoch with lowest val_loss
- `min_delta=0.001`: Minimum change to qualify as improvement

In [ ]:
# Task 4: EarlyStopping integration
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    min_delta=0.001,
    restore_best_weights=True,
    verbose=1
)

cnn_es = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
], name='CNN_EarlyStopping')

cnn_es.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Training CNN with EarlyStopping (max 30 epochs)...")
cnn_es_history = cnn_es.fit(
    x_train_norm, y_train,
    epochs=30,                    # Set high — EarlyStopping will cut it short
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

cnn_es_loss, cnn_es_acc = cnn_es.evaluate(x_test_norm, y_test, verbose=0)
actual_epochs = len(cnn_es_history.history['loss'])
print(f"\nStopped at epoch: {actual_epochs}/30")
print(f"CNN (EarlyStopping) Test Accuracy: {cnn_es_acc:.4f}")

---
## Final Comparison

In [ ]:
results = pd.DataFrame({
    'Model': [
        'ANN (Baseline, 10 epochs)',
        'CNN (Baseline, 10 epochs)',
        'CNN (Augmented, 10 epochs)',
        'ANN (Deep layout, 20 epochs)',
        'CNN (32→64→128 filters, 20 epochs)',
        'CNN (EarlyStopping, ≤30 epochs)'
    ],
    'Test Accuracy': [
        round(ann_test_acc, 4),
        round(cnn_test_acc, 4),
        round(aug_test_acc, 4),
        round(ann_deep_acc, 4),
        round(cnn_20ep_acc, 4),
        round(cnn_es_acc, 4)
    ],
    'Task Covered': [
        'Baseline ANN',
        'Baseline CNN',
        'Task 5 — Data Augmentation',
        'Task 1+3 — More layers + 20 epochs',
        'Task 2+3 — Filter scaling + 20 epochs',
        'Task 4 — EarlyStopping'
    ]
})

results['Test Accuracy %'] = (results['Test Accuracy'] * 100).round(2).astype(str) + '%'
print(results[['Model', 'Test Accuracy %', 'Task Covered']].to_string(index=False))

In [ ]:
# Visual bar chart of all model accuracies
fig, ax = plt.subplots(figsize=(12, 5))

colors = ['#e74c3c', '#2980b9', '#27ae60', '#e67e22', '#8e44ad', '#16a085']
bars = ax.bar(
    range(len(results)),
    results['Test Accuracy'] * 100,
    color=colors,
    edgecolor='white',
    linewidth=1.5,
    width=0.6
)

# Labels on bars
for bar, val in zip(bars, results['Test Accuracy']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f'{val*100:.2f}%',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

labels = [
    'ANN\nBaseline', 'CNN\nBaseline', 'CNN\nAugmented',
    'ANN Deep\n20ep', 'CNN Filters\n20ep', 'CNN Early\nStop'
]
ax.set_xticks(range(len(results)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('CIFAR-10: All Models Test Accuracy Comparison', fontsize=14, fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

---
# Conclusion

| Insight | Finding |
|--------|--------|
| **ANN vs CNN** | CNN significantly outperforms ANN because it preserves spatial structure via Conv2D kernels |
| **Filter scaling 32→64→128** | Progressive filter growth captures low→mid→high-level features in hierarchy |
| **Batch Normalization** | Stabilizes training, allows higher learning rates, acts as mild regularizer |
| **Dropout** | Prevents co-adaptation of neurons, improves generalization |
| **Data Augmentation** | Reduces overfitting by showing the model diverse transformations of the same image |
| **EarlyStopping** | Avoids over-training; restores best weights automatically |
| **More epochs** | Generally helps both ANN and CNN, though CNN benefits more from longer training |

### Why CNN beats ANN on image tasks:
1. **Weight sharing** — the same filter scans the whole image → fewer parameters
2. **Translation invariance** — detects a cat whether it's top-left or bottom-right
3. **Hierarchical features** — early layers = edges, deep layers = complex shapes
4. **Spatial context preserved** — neighboring pixels stay together, not destroyed by flattening